In [11]:
import pandas as pd
from langcodes import Language
import uuid
from datasets import load_dataset, Dataset

In [3]:
seed_df = load_dataset("ljvmiranda921/msde-seed-S1", split="train").to_pandas()
print(seed_df.columns)

Generating train split: 100%|██████████| 449017/449017 [00:00<00:00, 691487.96 examples/s]


Index(['id', 'source', 'language', 'strategy', 'source_id', 'prompt',
       'response'],
      dtype='object')


In [4]:
LANG_MAPPING = {"Tagalog": "tl", "Filipino": "fil"}

Process Cohere Aya Collection

In [8]:
def _iso3_to_iso2(code: str) -> str:
    tag = Language.get(code).to_tag()
    iso2 = tag.split("-")[0]
    return iso2

In [12]:
aya_ds = load_dataset("CohereLabs/aya_collection", "aya_dataset", split="train")
filtered_df = aya_ds.to_pandas()
filtered_df["language"] = filtered_df["language"].apply(_iso3_to_iso2)
filtered_df = filtered_df[filtered_df["language"].isin(set(LANG_MAPPING.values()))].reset_index(drop=True)  # fmt: skip
aya_df = pd.DataFrame(
    {
        "id": [uuid.uuid4().hex for _ in range(len(filtered_df))],
        "source": "CohereLabs/aya_collection",
        "language": filtered_df["language"].astype(str).values,
        "prompt": filtered_df["inputs"].astype(str).values,
        "response": filtered_df["targets"].astype(str).values,
        "strategy": [["generate", "respond"] for _ in range(len(filtered_df))],
        "source_id": filtered_df["id"].astype(str).values,
    }
)

In [35]:
len(aya_df)

1241

Process AllenAI Wildchat

In [36]:
wildchat_df_raw = pd.read_json("data/tl.jsonl.zst", lines=True, compression="zstd")

In [42]:
wildchat_df = pd.DataFrame(
    {
        # fmt: off
        "id": [uuid.uuid4().hex for _ in range(len(wildchat_df_raw))],
        "source": wildchat_df_raw["source"].astype(str).values,
        "language": "tl",
        "prompt": [row[0]["value"] for _, row in wildchat_df_raw.conversations.items()],
        "response": [row[1]["value"] for _, row in wildchat_df_raw.conversations.items()],
        "strategy": [["generate", "respond"] for _ in range(len(wildchat_df_raw))],
        "source_id": "agentlans/allenai-WildChat_" + wildchat_df_raw.index.astype(str),
        # fmt: on
    }
)

Process Saillab Alpaca Training Dataset

In [44]:
alpaca_ds = load_dataset("saillab/alpaca-filipino-cleaned", split="train")

Generating test split: 100%|██████████| 10401/10401 [00:00<00:00, 1350952.43 examples/s]


In [ ]:
alpaca_df_raw = alpaca_ds.to_pandas()
# If input is not NaN, append it to the instruction column
alpaca_df_raw["instruction"] = alpaca_df_raw.apply(
    lambda row: (
        row["instruction"]
        if pd.isna(row["input"])
        else f"{row['instruction']}\n\n{row['input']}"
    ),
    axis=1,
)
alpaca_df_raw

,instruction,input,output
0,Gawing wastong pangungusap ang pangungusap na...,Si Peter ay kumain ng mansanas,Si Pedro ay kumain ng mansanas.
1,Bumuo ng isang idyoma na may kaugnayan sa tag...,<walang input>,"""The sky's the limit"" - ibig sabihin ay walang..."
2,Tukuyin ang pinakamalaking lungsod sa Estados...,nan,"Ang pinakamalaking lungsod sa Estados Unidos, ..."
3,Bumuo ng limang positibong pagpapatibay\n\nnan,nan,1. Ikaw ay may kakayahang makamit ang kadakila...
4,Humanap ng 5 salita na naglalarawan sa damdam...,nan,"bigo, bigo, panghinaan ng loob, sama ng loob,..."
...,...,...,...
41596,Mag-imbento ng bagong pagkain na pinagsasama ...,Pineapple at spinach,Ang isang posibleng ulam na pinagsasama ang pi...
41597,Palitan ang pang-uri sa pangungusap ng angkop...,Nagsalita siya ng malakas.,Nagsalita siya ng malakas. (Ang pangungusap a...
41598,"Gamit ang ibinigay na larawan, tukuyin ang ur...",[Larawan],"Ikinalulungkot ko, ngunit ako ay isang modelo..."
41599,Bumuo ng isang dialogue tungkol sa mga kalama...,nan,"User: Hey AI, Pwede ba nating pag-usapan ang p..."


In [ ]:
alpaca_df = pd.DataFrame(
    {
        "id": [uuid.uuid4().hex for _ in range(len(alpaca_df_raw))],
        "source": "saillab/alpaca-filipino-cleaned",
        "language": "tl",
        "prompt": alpaca_df_raw["instruction"].astype(str).values,
        "response": alpaca_df_raw["output"].astype(str).values,
        "strategy": [["generate", "respond"] for _ in range(len(alpaca_df_raw))],
        "source_id": "saillab/alpaca-filipino-cleaned_"
        + alpaca_df_raw.index.astype(str),
    }
)
alpaca_df

,id,source,language,prompt,response,strategy,source_id
0,3665b25387354e4480e77b6263b92d73,saillab/alpaca-filipino-cleaned,tl,Gawing wastong pangungusap ang pangungusap na...,Si Pedro ay kumain ng mansanas.,"[generate, respond]",saillab/alpaca-filipino-cleaned_0
1,d3dd87d3b4104ce0b51f5bd33c6ed192,saillab/alpaca-filipino-cleaned,tl,Bumuo ng isang idyoma na may kaugnayan sa tag...,"""The sky's the limit"" - ibig sabihin ay walang...","[generate, respond]",saillab/alpaca-filipino-cleaned_1
2,06b30b7c60db4b02a549bae1d1fc3e43,saillab/alpaca-filipino-cleaned,tl,Tukuyin ang pinakamalaking lungsod sa Estados...,"Ang pinakamalaking lungsod sa Estados Unidos, ...","[generate, respond]",saillab/alpaca-filipino-cleaned_2
3,d4e566a992dc4e6a862f536e0f8dcdc5,saillab/alpaca-filipino-cleaned,tl,Bumuo ng limang positibong pagpapatibay\n\nnan,1. Ikaw ay may kakayahang makamit ang kadakila...,"[generate, respond]",saillab/alpaca-filipino-cleaned_3
4,23570c55c2004802a2bf27e4415f7509,saillab/alpaca-filipino-cleaned,tl,Humanap ng 5 salita na naglalarawan sa damdam...,"bigo, bigo, panghinaan ng loob, sama ng loob,...","[generate, respond]",saillab/alpaca-filipino-cleaned_4
...,...,...,...,...,...,...,...
41596,f071d48d7dc64d3bb0098a4f66d0653a,saillab/alpaca-filipino-cleaned,tl,Mag-imbento ng bagong pagkain na pinagsasama ...,Ang isang posibleng ulam na pinagsasama ang pi...,"[generate, respond]",saillab/alpaca-filipino-cleaned_41596
41597,b3b1864f18e34df1968ba9dec905b7ae,saillab/alpaca-filipino-cleaned,tl,Palitan ang pang-uri sa pangungusap ng angkop...,Nagsalita siya ng malakas. (Ang pangungusap a...,"[generate, respond]",saillab/alpaca-filipino-cleaned_41597
41598,8dbfdf51b6c949de9d0895ada291fd02,saillab/alpaca-filipino-cleaned,tl,"Gamit ang ibinigay na larawan, tukuyin ang ur...","Ikinalulungkot ko, ngunit ako ay isang modelo...","[generate, respond]",saillab/alpaca-filipino-cleaned_41598
41599,542ab37a4de943d2adc631b3b11e736d,saillab/alpaca-filipino-cleaned,tl,Bumuo ng isang dialogue tungkol sa mga kalama...,"User: Hey AI, Pwede ba nating pag-usapan ang p...","[generate, respond]",saillab/alpaca-filipino-cleaned_41599
